In [22]:
import pandas as pd
import numpy as np

print("=== Langkah 1: Memuat dan Menyiapkan Data ===")
# Pastikan file 'cardio_train.csv' berada di folder yang sama dengan notebook ini
try:
    df = pd.read_csv('cardio_train.csv', sep=';')
    print(f"Berhasil memuat {len(df)} baris data.")
except FileNotFoundError:
    print("Error: File 'cardio_train.csv' tidak ditemukan! Membuat data dummy 50.000+ untuk simulasi...")
    np.random.seed(42)
    df = pd.DataFrame({
        'age': np.random.randint(11000, 24000, size=55000),
        'weight': np.random.uniform(40, 120, size=55000),
        'height': np.random.uniform(140, 190, size=55000),
        'ap_hi': np.random.randint(90, 180, size=55000),
        'cardio': np.random.randint(0, 2, size=55000)
    })

# Preprocessing data metrik fisik
df['age_years'] = df['age'] / 365.25
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

# Mengambil sampel 5.000 data agar running time cepat (memenuhi syarat >5.000 data)
data_sample = df.head(5000).to_dict(orient='records')
print(f"Data siap diproses. Total sampel evaluasi: {len(data_sample)} baris.")

=== Langkah 1: Memuat dan Menyiapkan Data ===
Berhasil memuat 70000 baris data.
Data siap diproses. Total sampel evaluasi: 5000 baris.


In [23]:
def mu_segitiga(x, a, b, c):
    if x <= a or x >= c:
        return 0.0
    elif a < x <= b:
        return (x - a) / (b - a) if (b - a) != 0 else 1.0
    elif b < x < c:
        return (c - x) / (c - b) if (c - b) != 0 else 1.0
    return 0.0

def mu_trapesium(x, a, b, c, d):
    if x <= a or x >= d:
        return 0.0
    elif a < x <= b:
        return (x - a) / (b - a) if (b - a) != 0 else 1.0
    elif b < x <= c:
        return 1.0
    elif c < x < d:
        return (d - x) / (d - c) if (d - c) != 0 else 1.0
    return 0.0

In [24]:
def proses_fuzzifikasi(age_years, bmi, ap_hi):
    # Fuzzifikasi Variabel Umur
    u_muda = mu_trapesium(age_years, 0, 0, 35, 45)
    u_sedang = mu_segitiga(age_years, 40, 52, 65)
    u_tua = mu_trapesium(age_years, 60, 70, 100, 100)
    
    # Fuzzifikasi Variabel BMI
    bmi_kurang = mu_trapesium(bmi, 0, 0, 17, 18.5)
    bmi_normal = mu_segitiga(bmi, 18.0, 22.0, 25.0)
    bmi_lebih = mu_trapesium(bmi, 24.0, 28.0, 60, 60)
    
    # Fuzzifikasi Variabel Tekanan Darah Sistolik
    bp_normal = mu_trapesium(ap_hi, 0, 0, 115, 125)
    bp_tinggi = mu_trapesium(ap_hi, 120, 140, 300, 300)
    
    return {
        'umur': {'muda': u_muda, 'sedang': u_sedang, 'tua': u_tua},
        'bmi': {'kurang': bmi_kurang, 'normal': bmi_normal, 'lebih': bmi_lebih},
        'sistolik': {'normal': bp_normal, 'tinggi': bp_tinggi}
    }

In [25]:
def evaluasi_inferensi(fz):
    rules = []
    u = fz['umur']
    b = fz['bmi']
    s = fz['sistolik']
    
    # Memetakan kombinasi 15 Aturan sesuai kesepakatan tim
    rules.append(('Tinggi', min(u['tua'], b['lebih'], s['tinggi'])))         # Rule 1
    rules.append(('Tinggi', min(u['tua'], b['normal'], s['tinggi'])))        # Rule 2
    rules.append(('Tinggi', min(u['sedang'], b['lebih'], s['tinggi'])))       # Rule 3
    rules.append(('Sedang', min(u['sedang'], b['normal'], s['tinggi'])))     # Rule 4
    rules.append(('Tinggi', min(u['tua'], b['lebih'], s['normal'])))         # Rule 5
    rules.append(('Sedang', min(u['tua'], b['normal'], s['normal'])))        # Rule 6
    rules.append(('Rendah', min(u['muda'], b['normal'], s['normal'])))       # Rule 7
    rules.append(('Sedang', min(u['muda'], b['lebih'], s['tinggi'])))        # Rule 8
    rules.append(('Rendah', min(u['muda'], b['kurang'], s['normal'])))       # Rule 9
    rules.append(('Sedang', min(u['sedang'], b['kurang'], s['tinggi'])))     # Rule 10
    rules.append(('Sedang', min(u['sedang'], b['lebih'], s['normal'])))      # Rule 11
    rules.append(('Rendah', min(u['muda'], b['lebih'], s['normal'])))       # Rule 12
    rules.append(('Rendah', min(u['muda'], b['normal'], s['tinggi'])))       # Rule 13
    rules.append(('Sedang', min(u['sedang'], b['normal'], s['normal'])))     # Rule 14
    rules.append(('Tinggi', min(u['tua'], b['kurang'], s['tinggi'])))        # Rule 15
    
    return rules

In [26]:
def defuzz_mamdani(rules):
    agregat_rendah = max([val for kat, val in rules if kat == 'Rendah'], default=0)
    agregat_sedang = max([val for kat, val in rules if kat == 'Sedang'], default=0)
    agregat_tinggi = max([val for kat, val in rules if kat == 'Tinggi'], default=0)
    
    pembilang = 0.0
    penyebut = 0.0
    
    # Diskretisasi area output 0-100 dengan step 2
    for z in range(0, 101, 2):
        mu_r = min(agregat_rendah, mu_trapesium(z, 0, 0, 25, 45))
        mu_s = min(agregat_sedang, mu_segitiga(z, 35, 50, 65))
        mu_t = min(agregat_tinggi, mu_trapesium(z, 55, 75, 100, 100))
        
        mu_z = max(mu_r, mu_s, mu_t)
        
        pembilang += z * mu_z
        penyebut += mu_z
        
    return pembilang / penyebut if penyebut != 0 else 0.0

In [27]:
def defuzz_sugeno(rules):
    # Singleton konstan untuk Sugeno Orde-0
    nilai_konstanta = {'Rendah': 15.0, 'Sedang': 50.0, 'Tinggi': 85.0}
    
    pembilang = 0.0
    penyebut = 0.0
    
    for kategori, alpha_pred in rules:
        pembilang += alpha_pred * nilai_konstanta[kategori]
        penyebut += alpha_pred
        
    return pembilang / penyebut if penyebut != 0 else 0.0

In [29]:
print("=== Langkah 2: Menjalankan Komputasi Sistem Fuzzy ===")
prediksi_mamdani = []
prediksi_sugeno = []
ground_truth = []

for row in data_sample:
    # Eksekusi pipeline fuzzy
    fz = proses_fuzzifikasi(row['age_years'], row['bmi'], row['ap_hi'])
    rules = evaluasi_inferensi(fz)
    
    # Defuzzifikasi & Thresholding biner
    out_mamdani = defuzz_mamdani(rules)
    pred_m = 1 if out_mamdani > 50 else 0
    prediksi_mamdani.append(pred_m)
    
    out_sugeno = defuzz_sugeno(rules)
    pred_s = 1 if out_sugeno > 50 else 0
    prediksi_sugeno.append(pred_s)
    
    ground_truth.append(row['cardio'])

# Perhitungan Akurasi Akhir secara manual
acc_mamdani = sum([1 for m, gt in zip(prediksi_mamdani, ground_truth) if m == gt]) / len(ground_truth)
acc_sugeno = sum([1 for s, gt in zip(prediksi_sugeno, ground_truth) if s == gt]) / len(ground_truth)

print("\n=== HASIL EVALUASI PERFORMA CARDIO ===")
print(f"Total baris data yang dievaluasi : {len(ground_truth)} baris")
print(f"Akurasi Akhir Sistem Mamdani     : {acc_mamdani * 100:.2f}%")
print(f"Akurasi Akhir Sistem Sugeno      : {acc_sugeno * 100:.2f}%")

=== Langkah 2: Menjalankan Komputasi Sistem Fuzzy ===

=== HASIL EVALUASI PERFORMA CARDIO ===
Total baris data yang dievaluasi : 5000 baris
Akurasi Akhir Sistem Mamdani     : 67.28%
Akurasi Akhir Sistem Sugeno      : 67.56%
